In [2]:
import os
BASE_DIR = "\\notebooks\\experiments"
if os.getcwd().endswith(BASE_DIR):
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [3]:
from src.models.model_handler import save_model, load_model, get_class_def_from_notebook
from src.models.testing_utils import evaluate_continuous, predict_continuous
from src.data.preprocessing import generate_classification_tensors
from src.data.DatasetGenerator import DatasetGenerator
from src.models.training_utils import train_regressor_model
from sklearn.preprocessing import LabelEncoder
from src.utils.helpers import load_param

from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
import numpy as np
import torch

In [3]:
dg = DatasetGenerator(num_particules=10000)

generators = [
    ("rw", dg.generate_RandomWalk),
    ("bm", dg.generate_BrownianMotion),
    ("ctrw", dg.generate_ContinuousTimeRandomWalk),
    ("fbm", dg.generate_FractionalBrownianMotion),
    ("lw", dg.generate_LevyWalks),
    ("attm", dg.generate_AnnealedTransitTimeModel),
    ("sbm", dg.generate_ScaledBrownianMotion),
]

dataframes = []
for prefix, func in generators:
    df = func(random_params=True, num_passos=200)
    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True)
data

c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\fbm\fbm.py:171: UserWarning: Combination of increments n and Hurst value H invalid for Davies-Harte method. Reverting to Hosking method. Occurs when n is small and Hurst is close to 1. 
  warnings.warn(


,coords,num_steps,trajectory_type,attributes
0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",200,Random Walk,{'movement_type': 'hard_stop'}
1,"[[0.0, 0.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 0.0,...",200,Random Walk,{'movement_type': 'nonstop'}
2,"[[0.0, 0.0, 0.0, 0.0, -1.0, -1.0, -2.0, -2.0, ...",200,Random Walk,{'movement_type': 'nonstop'}
3,"[[0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0...",200,Random Walk,{'movement_type': 'chill_stop'}
4,"[[0.0, 1.0, 1.0, 2.0, 3.0, 2.0, 3.0, 4.0, 3.0,...",200,Random Walk,{'movement_type': 'nonstop'}
...,...,...,...,...
69995,"[[-0.6979899289758802, -1.621985164400844, -1....",200,Scaled Brownian Motion,{'beta': 0.783805651920362}
69996,"[[-0.5657484695122497, -0.29379565591873397, 0...",200,Scaled Brownian Motion,{'beta': 1.401982130582025}
69997,"[[0.24100944983094946, 0.4893597392352079, -0....",200,Scaled Brownian Motion,{'beta': 0.5926571229118587}
69998,"[[-1.098077922641129, -2.0098070961657344, -2....",200,Scaled Brownian Motion,{'beta': 1.1539733707564794}


In [4]:
le = LabelEncoder()
X_tensor, y_tensor = generate_classification_tensors(data, 'trajectory_type', le)

In [5]:
# Aconseguir X_train, X_val, X_test, y_train, y_val i y_test
n, train_end, val_end = len(X_tensor), int(0.7 * len(X_tensor)), int(0.85 * len(X_tensor))
X_train, X_val, X_test = X_tensor[:train_end], X_tensor[train_end:val_end], X_tensor[val_end:]
y_train, y_val, y_test = y_tensor[:train_end], y_tensor[train_end:val_end], y_tensor[val_end:]

In [6]:
# Muntem els Data_loaders
learning_batch_size = 32
try_batch_size = 1

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=learning_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=try_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=try_batch_size, shuffle=True)

In [11]:
# 6. Definir la arquitectura CNN
class TrajectoryTypeClassifier(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c

input_channels = 2

# Inicialitzem el model d'aquesta manera per poder després guardar facilment els paràmetres
model_params = {
    "input_channels": input_channels,
    "num_classes": len(le.classes_)
}

model = TrajectoryTypeClassifier(**model_params)

In [12]:
# optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, 
#     mode='min',  # Minimizar val_loss
#     factor=0.2,  # Reduce LR a 20% del actual
#     patience=3,  # Espera 3 epochs sin mejora
#     verbose=True  # Muestra mensajes
# )


# best_model = train_regressor_model(
#     model=model,
#     criterion=nn.CrossEntropyLoss(),
#     optimizer=optimizer,
#     num_epochs=15,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     scheduler=scheduler,
#     print_epoch=True,
#     patience=5
# )

# 8. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 9. Entrenar el modelo
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/15], Loss: 1.4550
Epoch [2/15], Loss: 1.3147
Epoch [3/15], Loss: 0.9311
Epoch [4/15], Loss: 0.6401
Epoch [5/15], Loss: 0.5744
Epoch [6/15], Loss: 0.5499
Epoch [7/15], Loss: 0.5122
Epoch [8/15], Loss: 0.4826
Epoch [9/15], Loss: 0.4412
Epoch [10/15], Loss: 0.7668
Epoch [11/15], Loss: 0.4279
Epoch [12/15], Loss: 0.3895
Epoch [13/15], Loss: 0.3841
Epoch [14/15], Loss: 0.3762
Epoch [15/15], Loss: 0.3583


In [13]:
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Índices de las clases predichas
    true_labels = torch.argmax(y_test, dim=1)  # Convertir one-hot a índices
    accuracy = (predicted == true_labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.8814


In [108]:
import random

# Seleccionar el primer ejemplo de X_test y añadir una dimensión batch
random_value = random.randint(1, 500)
# print("Valor random entre 1 y 500:", random_value)
first_example = X_test[random_value].unsqueeze(0)  # shape: (1, 2, 200)

# Pasar por el modelo
model.eval()
with torch.no_grad():
    output = model(first_example)
    predicted_class = torch.argmax(output, dim=1).item()

# Mostrar el resultado
print("Predicción (índice de clase):", predicted_class)
print("Clase predicha:", le.inverse_transform([predicted_class])[0])
# Mostrar la clase real
true_class = torch.argmax(y_test[random_value]).item()
print("Clase real:", le.inverse_transform([true_class])[0])

Predicción (índice de clase): 5
Clase predicha: Random Walk
Clase real: Random Walk


In [11]:
evaluate_continuous(best_model, X_test, y_test)

Test MSE: 0.0189


0.018906662240624428

In [12]:
predict_continuous(best_model, X_test, y_test, num_samples=10)

Predicciones:
Ejemplo 1: Predicho = 0.6406972408294678, Real = [0.54004186]
Ejemplo 2: Predicho = 0.4589676260948181, Real = [0.52591753]
Ejemplo 3: Predicho = 0.3457889258861542, Real = [0.17001842]
Ejemplo 4: Predicho = 0.7203972935676575, Real = [0.77847546]
Ejemplo 5: Predicho = 0.27398133277893066, Real = [0.2676552]
Ejemplo 6: Predicho = 0.5036356449127197, Real = [0.36038285]
Ejemplo 7: Predicho = 0.6786308884620667, Real = [0.8246553]
Ejemplo 8: Predicho = 0.6253182888031006, Real = [0.70867896]
Ejemplo 9: Predicho = 0.9011772871017456, Real = [0.963228]
Ejemplo 10: Predicho = 0.7878854274749756, Real = [0.96272326]


In [13]:
SAVE_PATH = "../../../models/trained_models/"

In [110]:
save_model(model_instance=model,
           model_source=get_class_def_from_notebook(model.__class__.__name__),
           attributes=model_params,
           input_channels=input_channels,
           path=load_param("path", "model")
           )

In [5]:
prova_carrega_model=load_model(
    load_param("path", "model") +
    "TrajectoryTypeClassifier.pth"
)

NameError: name 'le' is not defined